# 03 — Reranker Comparison

**Question.** Which local model below 1B parameters best reorders the same frozen hybrid candidate pool?

Cross-encoders and the 0.6B Qwen reranker are evaluated on identical query–article contexts.


## Setup


In [ ]:
from pathlib import Path
import os, sys, json, time, copy, itertools, importlib.util, subprocess

def locate_project(start=Path.cwd()):
    for parent in [start.resolve(), *start.resolve().parents]:
        if (parent / "pyproject.toml").exists() and (parent / "src/avito_retriever").exists():
            return parent
    candidate = Path("/content/avito-retriever")
    if candidate.exists():
        return candidate
    raise FileNotFoundError("Clone/open avito-retriever or change PROJECT_ROOT in this cell")

PROJECT_ROOT = locate_project()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

IN_COLAB = module_available("google.colab")
if IN_COLAB and os.environ.get("AVITO_AUTO_INSTALL", "1") != "0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{PROJECT_ROOT}[lexical,neural,dev]",
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from avito_retriever.experiments.notebook import resolve_data_dir, result_dir, highlight_best, save_json, save_yaml

DATA_DIR = resolve_data_dir(PROJECT_ROOT)
SEED = 42
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)
print("Data:", DATA_DIR)


In [ ]:
from avito_retriever.data.io import load_calibration
from avito_retriever.evaluation.metrics import evaluate_rankings
from avito_retriever.evaluation.selection import metric_on_split
from avito_retriever.reranking.bi_encoder import rerank_with_bi_encoder
from avito_retriever.reranking.cross_encoder import rerank_with_cross_encoder

OUT = result_dir(PROJECT_ROOT, "03_reranker_comparison")
PREV = result_dir(PROJECT_ROOT, "02_dense_knn_fusion_search")
LEX = result_dir(PROJECT_ROOT, "01_sentencepiece_bm25f_search")
calibration = load_calibration(DATA_DIR)
split = pd.read_csv(LEX / "selection_split.csv")
hybrid = pd.read_parquet(PREV / "best_hybrid_rankings.parquet")
candidates = hybrid[hybrid["rank"] <= 50].copy()
contexts_frame = pd.read_parquet(PREV / "reranker_contexts.parquet")
contexts = {(int(r.query_id), int(r.article_id)): r.text for r in contexts_frame.itertuples(index=False)}
CROSS_ENCODERS = ["DiTy/cross-encoder-russian-msmarco", "BAAI/bge-reranker-v2-m3",
                  "Alibaba-NLP/gte-multilingual-reranker-base"]
BI_ENCODERS = ["intfloat/multilingual-e5-base",
               "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"]
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"


## Bi-encoder reranker sweep
These models score the same top-50 query–context pairs independently, so they are compared fairly with the cross-encoders.


In [ ]:
rows, rankings_by_model = [], {}
for model_name in BI_ENCODERS:
    started = time.perf_counter()
    try:
        ranking = rerank_with_bi_encoder(calibration, candidates, contexts, model_name,
                                          batch_size=64 if DEVICE=="cuda" else 16,
                                          device=DEVICE, source=model_name)
        metrics, per_query = evaluate_rankings(ranking, calibration)
        trial = model_name.split("/")[-1]
        ranking.to_parquet(OUT / f"rankings_{trial}.parquet", index=False)
        per_query.to_parquet(OUT / f"per_query_{trial}.parquet", index=False)
        rankings_by_model[model_name] = ranking
        rows.append({"model": model_name, "kind": "bi_encoder",
                     "tune_map@10": metric_on_split(per_query, split, "ap@10", "tune"),
                     "seconds": time.perf_counter()-started, **metrics})
    except Exception as error:
        rows.append({"model": model_name, "kind": "bi_encoder", "status": f"error: {error}"})
pd.DataFrame(rows).to_csv(OUT / "reranker_progress.csv", index=False)
display(pd.DataFrame(rows))


## Cross-encoder sweep


In [ ]:
for model_name in CROSS_ENCODERS:
    started = time.perf_counter()
    try:
        ranking = rerank_with_cross_encoder(calibration, candidates, contexts, model_name,
                                            batch_size=32 if DEVICE=="cuda" else 8, max_length=512,
                                            device=DEVICE, source=model_name)
        metrics, per_query = evaluate_rankings(ranking, calibration)
        trial = model_name.split("/")[-1]
        ranking.to_parquet(OUT / f"rankings_{trial}.parquet", index=False)
        per_query.to_parquet(OUT / f"per_query_{trial}.parquet", index=False)
        rankings_by_model[model_name] = ranking
        rows.append({"model": model_name, "kind": "cross_encoder",
                     "tune_map@10": metric_on_split(per_query, split, "ap@10", "tune"),
                     "seconds": time.perf_counter()-started, **metrics})
    except Exception as error:
        rows.append({"model": model_name, "kind": "cross_encoder", "status": f"error: {error}"})
    pd.DataFrame(rows).to_csv(OUT / "reranker_progress.csv", index=False)
table = pd.DataFrame(rows).dropna(subset=["tune_map@10"]).sort_values("tune_map@10", ascending=False).reset_index(drop=True)
assert not table.empty, "Every reranker failed; inspect reranker_progress.csv"
display(highlight_best(table, ["tune_map@10", "map@10", "recall@20"]))


## Optional Qwen3-Reranker-0.6B
Set `RUN_QWEN=True` on a GPU runtime. The cell uses the model's yes/no relevance probability.


In [ ]:
RUN_QWEN = DEVICE == "cuda"
QWEN_MODEL = "Qwen/Qwen3-Reranker-0.6B"

def qwen_scores(pairs, model_name=QWEN_MODEL, batch_size=8, max_length=2048):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,
                                                  device_map="auto").eval()
    no_id, yes_id = tokenizer.convert_tokens_to_ids("no"), tokenizer.convert_tokens_to_ids("yes")
    system = 'Judge whether the Document meets the requirements based on the Query. Answer only "yes" or "no".'
    values = []
    for start in range(0, len(pairs), batch_size):
        prompts = [f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n<Query>: {q}\n<Document>: {d}<|im_end|>\n<|im_start|>assistant\n"
                   for q,d in pairs[start:start+batch_size]]
        batch = tokenizer(prompts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(model.device)
        with torch.no_grad(): logits = model(**batch).logits[:, -1, [no_id, yes_id]]
        values.extend(torch.softmax(logits.float(), dim=-1)[:,1].cpu().tolist())
    return np.asarray(values)

if RUN_QWEN:
    try:
        qmap = dict(zip(calibration.query_id.astype(int), calibration.query_text.astype(str)))
        ordered = candidates.sort_values(["query_id","rank"]).reset_index(drop=True)
        pairs = [(qmap[int(r.query_id)], contexts[(int(r.query_id),int(r.article_id))]) for r in ordered.itertuples()]
        ordered["score"] = qwen_scores(pairs); ordered["source"] = QWEN_MODEL
        ordered["rank"] = ordered.groupby("query_id").score.rank(method="first", ascending=False).astype(int)
        qwen_ranking = ordered[["query_id","article_id","score","rank","source"]].sort_values(["query_id","rank"])
        metrics, per_query = evaluate_rankings(qwen_ranking, calibration)
        qwen_ranking.to_parquet(OUT / "rankings_Qwen3-Reranker-0.6B.parquet", index=False)
        per_query.to_parquet(OUT / "per_query_Qwen3-Reranker-0.6B.parquet", index=False)
        rows.append({"model": QWEN_MODEL, "kind": "llm_reranker",
                     "tune_map@10": metric_on_split(per_query, split, "ap@10", "tune"), **metrics})
        rankings_by_model[QWEN_MODEL] = qwen_ranking
    except Exception as error:
        rows.append({"model": QWEN_MODEL, "kind": "llm_reranker", "status": f"error: {error}"})
table = pd.DataFrame(rows).dropna(subset=["tune_map@10"]).sort_values("tune_map@10", ascending=False).reset_index(drop=True)
display(highlight_best(table, ["tune_map@10", "map@10"]))


## Freeze winner and report confirm


In [ ]:
best = table.iloc[0].to_dict(); selected = rankings_by_model[best["model"]]
metrics, per_query = evaluate_rankings(selected, calibration)
selected.to_parquet(OUT / "best_reranked_rankings.parquet", index=False)
per_query.to_parquet(OUT / "best_reranked_per_query.parquet", index=False)
best["confirm_map@10"] = metric_on_split(per_query, split, "ap@10", "confirm")
table.to_csv(OUT / "reranker_leaderboard.csv", index=False); save_json(best, OUT / "best_reranker_config.json")
display(pd.DataFrame([best]))


## Interpretation
The winner is selected by tune MAP@10. Notebook 05 tests whether its confirm-set improvement over baseline is statistically credible.
